In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import pickle

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
df=pd.read_csv(r"/content/IMDB-Movie-Dataset(2023-1951).csv")
df.shape

(2199, 8)

In [6]:
df.head()

,Unnamed: 0,movie_id,movie_name,year,genre,overview,director,cast
0,0,tt15354916,Jawan,2023,"Action, Thriller",A high-octane action thriller which outlines t...,Atlee,"Shah Rukh Khan, Nayanthara, Vijay Sethupathi, ..."
1,1,tt15748830,Jaane Jaan,2023,"Crime, Drama, Mystery",A single mother and her daughter who commit a ...,Sujoy Ghosh,"Kareena Kapoor, Jaideep Ahlawat, Vijay Varma, ..."
2,2,tt11663228,Jailer,2023,"Action, Comedy, Crime",A retired jailer goes on a manhunt to find his...,Nelson Dilipkumar,"Rajinikanth, Mohanlal, Shivarajkumar, Jackie S..."
3,3,tt14993250,Rocky Aur Rani Kii Prem Kahaani,2023,"Comedy, Drama, Family",Flamboyant Punjabi Rocky and intellectual Beng...,Karan Johar,"Ranveer Singh, Alia Bhatt, Dharmendra, Shabana..."
4,4,tt15732324,OMG 2,2023,"Comedy, Drama",An unhappy civilian asks the court to mandate ...,Amit Rai,"Pankaj Tripathi, Akshay Kumar, Yami Gautam, Pa..."


In [7]:
df.describe()

,Unnamed: 0
count,2199.000000
mean,1099.369259
std,635.344495
min,0.000000
25%,549.500000
50%,1099.000000
75%,1649.500000
max,2199.000000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2199 entries, 0 to 2198
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  2199 non-null   int64 
 1   movie_id    2199 non-null   object
 2   movie_name  2199 non-null   object
 3   year        2134 non-null   object
 4   genre       2199 non-null   object
 5   overview    2199 non-null   object
 6   director    2199 non-null   object
 7   cast        2199 non-null   object
dtypes: int64(1), object(7)
memory usage: 137.6+ KB


In [9]:
df.columns

Index(['Unnamed: 0', 'movie_id', 'movie_name', 'year', 'genre', 'overview',
       'director', 'cast'],
      dtype='object')

In [10]:
df.drop(['Unnamed: 0'],inplace=True,axis=1)

In [11]:
df.head()

,movie_id,movie_name,year,genre,overview,director,cast
0,tt15354916,Jawan,2023,"Action, Thriller",A high-octane action thriller which outlines t...,Atlee,"Shah Rukh Khan, Nayanthara, Vijay Sethupathi, ..."
1,tt15748830,Jaane Jaan,2023,"Crime, Drama, Mystery",A single mother and her daughter who commit a ...,Sujoy Ghosh,"Kareena Kapoor, Jaideep Ahlawat, Vijay Varma, ..."
2,tt11663228,Jailer,2023,"Action, Comedy, Crime",A retired jailer goes on a manhunt to find his...,Nelson Dilipkumar,"Rajinikanth, Mohanlal, Shivarajkumar, Jackie S..."
3,tt14993250,Rocky Aur Rani Kii Prem Kahaani,2023,"Comedy, Drama, Family",Flamboyant Punjabi Rocky and intellectual Beng...,Karan Johar,"Ranveer Singh, Alia Bhatt, Dharmendra, Shabana..."
4,tt15732324,OMG 2,2023,"Comedy, Drama",An unhappy civilian asks the court to mandate ...,Amit Rai,"Pankaj Tripathi, Akshay Kumar, Yami Gautam, Pa..."


In [12]:
i=0
for i in df.columns:
  print(i,df[i].nunique())

movie_id 2199
movie_name 2161
year 84
genre 225
overview 2120
director 1064
cast 2178


In [13]:
df=df.drop_duplicates(['movie_name'])

In [14]:
df.isnull().sum()

,0
movie_id,0
movie_name,0
year,65
genre,0
overview,0
director,0
cast,0


In [15]:
df['year']=pd.to_numeric(df['year'],errors='coerce')

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2161 entries, 0 to 2198
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   movie_id    2161 non-null   object 
 1   movie_name  2161 non-null   object 
 2   year        2087 non-null   float64
 3   genre       2161 non-null   object 
 4   overview    2161 non-null   object 
 5   director    2161 non-null   object 
 6   cast        2161 non-null   object 
dtypes: float64(1), object(6)
memory usage: 135.1+ KB


In [17]:
mean=df.year.mean()

In [18]:
mean

np.float64(2007.731672256828)

In [19]:
round(mean)

2008

In [20]:
# But year has nothing to do therefore we will leave as it as out model
# is trained on cast, overview, genre and director

In [21]:
df=df.dropna()

In [22]:
df['year']=df['year'].astype('int')

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2087 entries, 0 to 2198
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   movie_id    2087 non-null   object
 1   movie_name  2087 non-null   object
 2   year        2087 non-null   int64 
 3   genre       2087 non-null   object
 4   overview    2087 non-null   object
 5   director    2087 non-null   object
 6   cast        2087 non-null   object
dtypes: int64(1), object(6)
memory usage: 130.4+ KB


In [24]:
df.shape

(2087, 7)

In [25]:
df.isna().sum()

,0
movie_id,0
movie_name,0
year,0
genre,0
overview,0
director,0
cast,0


In [26]:
df['tags'] = (
    "Genre: " + df['genre'] + ". "
    "Director: " + df['director'] + ". "
    "Cast: " + df['cast'] + ". "
    "Overview: " + df['overview']
)

df['tags'] = df['tags'].str.lower()

In [27]:
df.tags[0]

'genre: action, thriller. director: atlee. cast: shah rukh khan, nayanthara, vijay sethupathi, deepika padukone. overview: a high-octane action thriller which outlines the emotional journey of a man who is set to rectify the wrongs in the society.'

In [28]:
scaler=MinMaxScaler()

In [29]:
df['popularity_score']=scaler.fit_transform(df[['year']])

In [30]:
df.head()

,movie_id,movie_name,year,genre,overview,director,cast,tags,popularity_score
0,tt15354916,Jawan,2023,"Action, Thriller",A high-octane action thriller which outlines t...,Atlee,"Shah Rukh Khan, Nayanthara, Vijay Sethupathi, ...","genre: action, thriller. director: atlee. cast...",0.968085
1,tt15748830,Jaane Jaan,2023,"Crime, Drama, Mystery",A single mother and her daughter who commit a ...,Sujoy Ghosh,"Kareena Kapoor, Jaideep Ahlawat, Vijay Varma, ...","genre: crime, drama, mystery. director: sujoy ...",0.968085
2,tt11663228,Jailer,2023,"Action, Comedy, Crime",A retired jailer goes on a manhunt to find his...,Nelson Dilipkumar,"Rajinikanth, Mohanlal, Shivarajkumar, Jackie S...","genre: action, comedy, crime. director: nelson...",0.968085
3,tt14993250,Rocky Aur Rani Kii Prem Kahaani,2023,"Comedy, Drama, Family",Flamboyant Punjabi Rocky and intellectual Beng...,Karan Johar,"Ranveer Singh, Alia Bhatt, Dharmendra, Shabana...","genre: comedy, drama, family. director: karan ...",0.968085
4,tt15732324,OMG 2,2023,"Comedy, Drama",An unhappy civilian asks the court to mandate ...,Amit Rai,"Pankaj Tripathi, Akshay Kumar, Yami Gautam, Pa...","genre: comedy, drama. director: amit rai. cast...",0.968085


In [31]:
df.sort_values(by='popularity_score',ascending=True)

,movie_id,movie_name,year,genre,overview,director,cast,tags,popularity_score
1293,tt0233938,Indrasabha,1932,"Musical, Romance",Big-budget movie with over 69 songs. 'The Hind...,Jamahedji Jehangirji Madan,"Nissar, Jehanara Kajjan, Abdul Rehman Kabuli, ...","genre: musical, romance. director: jamahedji j...",0.000000
1580,tt0043306,Awaara,1951,"Drama, Musical, Romance",A poor young man named Raj joins a criminal ga...,Raj Kapoor,"Raj Kapoor, Nargis, Prithviraj Kapoor, K.N. Singh","genre: drama, musical, romance. director: raj ...",0.202128
768,tt0045693,Do Bigha Zamin,1953,Drama,In the hope of earning enough money to pay off...,Bimal Roy,"Balraj Sahni, Nirupa Roy, Ratan Kumar, Murad",genre: drama. director: bimal roy. cast: balra...,0.223404
2192,tt0047561,Taxi Driver,1954,"Musical, Romance","Mangal drives a taxi by day, then drinks at ni...",Chetan Anand,"Dev Anand, Kalpana Kartik, Sheila Ramani, John...","genre: musical, romance. director: chetan anan...",0.234043
1616,tt0267289,Baap Beti,1954,Drama,Add a Plot,Bimal Roy,"Sunalini Devi, Dolly, Tabassum Govil, Nasir Hu...",genre: drama. director: bimal roy. cast: sunal...,0.234043
...,...,...,...,...,...,...,...,...,...
251,tt21383812,The Crew,2024,"Comedy, Drama",Follows three hard-working women as their dest...,Rajesh Krishnan,"Kareena Kapoor, Kriti Sanon, Rajkummar Rao, Tabu","genre: comedy, drama. director: rajesh krishna...",0.978723
821,tt7790918,Untitled,2024,Action,"In the dark underbelly of the city, Bloodshed ...",Vishnuvardhan,"Kiara Advani, Nikitin Dheer, Anupam Kher, Such...",genre: action. director: vishnuvardhan. cast: ...,0.978723
1773,tt11203296,Khiladi 1080,2025,Action,"Four identical brothers Raghupati, Raghav, Raj...",Kabir Sadanand,"Rakul Preet Singh, Sonakshi Sinha, Huma Quresh...",genre: action. director: kabir sadanand. cast:...,0.989362
294,tt5885564,Don 3: The Final Chapter,2025,"Action, Thriller",Plot under wraps.,Farhan Akhtar,Ranveer Singh,"genre: action, thriller. director: farhan akht...",0.989362


In [32]:
df=df[df['year']<=2025]

In [33]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1,2)
)

In [34]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [35]:
tfidf_similarity = cosine_similarity(tfidf_matrix)

In [36]:
tfidf_similarity

array([[1.        , 0.01793673, 0.01535898, ..., 0.0037057 , 0.00318385,
        0.01325386],
       [0.01793673, 1.        , 0.01356145, ..., 0.00328399, 0.01599404,
        0.00416145],
       [0.01535898, 0.01356145, 1.        , ..., 0.00363652, 0.0159261 ,
        0.03734386],
       ...,
       [0.0037057 , 0.00328399, 0.00363652, ..., 1.        , 0.00307179,
        0.00313809],
       [0.00318385, 0.01599404, 0.0159261 , ..., 0.00307179, 1.        ,
        0.05594785],
       [0.01325386, 0.00416145, 0.03734386, ..., 0.00313809, 0.05594785,
        1.        ]])

In [37]:
df.shape

(2086, 9)

In [38]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [39]:
embeddings=model.encode(
    df['tags'].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

In [40]:
embedding_similarity=cosine_similarity(embeddings)

In [41]:
len(embedding_similarity)

2086

In [42]:
hybrid_similarity=(
    0.6*tfidf_similarity
    +0.4*embedding_similarity
)

In [43]:
def recommend(movie_name, top_n=10):

    movie_index = df[df['movie_name'] == movie_name].index[0]

    similarity_scores = list(enumerate(hybrid_similarity[movie_index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:50]

    movie_indices = [i[0] for i in similarity_scores]

    candidates = df.iloc[movie_indices].copy()

    similarity_values = [score for _, score in similarity_scores]

    candidates['final_score'] = (
        0.7 * np.array(similarity_values) +
        0.3 * candidates['popularity_score']
    )

    recommendations = candidates.sort_values(
        'final_score',
        ascending=False
    ).head(top_n)

    return recommendations[['movie_name', 'year', 'genre']]

In [49]:
print(recommend("PK"))

                movie_name  year                      genre
24                   Dunki  2023              Comedy, Drama
1491               Ayalaan  2024  Action, Adventure, Sci-Fi
1980           Love Nation  2023              Drama, Sci-Fi
10                3 Idiots  2009              Comedy, Drama
97     Munna Bhai M.B.B.S.  2003              Comedy, Drama
523   Lage Raho Munna Bhai  2006     Comedy, Drama, Romance
139                  Sanju  2018   Biography, Comedy, Drama
751                  Joker  2012     Comedy, Family, Sci-Fi
284            Housefull 5  2024              Comedy, Drama
1589                 Cargo  2019     Drama, Fantasy, Sci-Fi


In [45]:
pickle.dump(df, open("movies_metadata.pkl", "wb"))

In [46]:
pickle.dump(hybrid_similarity, open("movie_similarity.pkl", "wb"))